In [1]:
import json
import pandas as pd
import numpy as np

with open('data/train/inputs-base.json', 'r', encoding='utf-8') as f:
    inputs_data = json.load(f)

with open('data/train/outcomes.json', 'r', encoding='utf-8') as f:
    outcomes_data = json.load(f)

rows = []
for ep in outcomes_data['episodes']:
    ep_id = ep['episode_id']
    models = ep['models']
    
    row = {'episode_id': ep_id}
    for model_name in ['ax31-light', 'ax31', 'axk1-think']:
        m_info = models.get(model_name, {})
        row[f'{model_name}_score'] = m_info.get('score', 0.0)
        row[f'{model_name}_input_tokens'] = m_info.get('input_tokens', 0)
        row[f'{model_name}_output_tokens'] = m_info.get('output_tokens', 0)
        row[f'{model_name}_cost'] = m_info.get('cost', 0.0)
    
    rows.append(row)

df = pd.DataFrame(rows)
print(f"총 분석 문항 수: {len(df)}")
df.head()

총 분석 문항 수: 1760


,episode_id,ax31-light_score,ax31-light_input_tokens,ax31-light_output_tokens,ax31-light_cost,ax31_score,ax31_input_tokens,ax31_output_tokens,ax31_cost,axk1-think_score,axk1-think_input_tokens,axk1-think_output_tokens,axk1-think_cost
0,train-0001,0,112,18,0.0,0.5,110,229,0.0,1,122,4747,0.0
1,train-0002,1,646,68,0.0,1,610,80,0.0,1,520,4883,0.0
2,train-0003,0,352,121,0.0,0,346,244,0.0,0,336,3940,0.0
3,train-0004,1,168,543,0.0,0.5,164,424,0.0,0,166,1498,0.0
4,train-0005,1,524,34,0.0,1,488,28,0.0,1,428,998,0.0


In [5]:
with open('data/train/outcomes.json', 'r', encoding='utf-8') as f:
    outcomes_data = json.load(f)

MODELS = ['ax31-light', 'ax31', 'axk1-think']
rows = []

for ep in outcomes_data['episodes']:
    ep_id = ep['episode_id']
    models_info = ep['models']
    
    row = {'episode_id': ep_id}
    for m in MODELS:
        m_data = models_info.get(m, {})
        row[f'{m}_score'] = m_data.get('score', 0.0)
        row[f'{m}_input_tokens'] = m_data.get('input_tokens', 0)
        row[f'{m}_output_tokens'] = m_data.get('output_tokens', 0)
        
    rows.append(row)

df = pd.DataFrame(rows)

PRICES = {
    'ax31-light': {'input': 1.0, 'output': 1.0},
    'ax31':       {'input': 2.0, 'output': 2.0},
    'axk1-think': {'input': 5.0, 'output': 5.0}
}

for m in MODELS:
    df[f'{m}_score'] = pd.to_numeric(df[f'{m}_score'], errors='coerce').fillna(0.0)
    df[f'{m}_input_tokens'] = pd.to_numeric(df[f'{m}_input_tokens'], errors='coerce').fillna(0)
    df[f'{m}_output_tokens'] = pd.to_numeric(df[f'{m}_output_tokens'], errors='coerce').fillna(0)

    p_in = PRICES[m]['input']
    p_out = PRICES[m]['output']
    df[f'{m}_cost'] = (df[f'{m}_input_tokens'] * p_in) + (df[f'{m}_output_tokens'] * p_out)

base_total_cost = df['ax31-light_cost'].sum()

print(f"ax31-light 총 비용(Base Total Cost): {base_total_cost:,.2f}")

summary = []
for m in MODELS:
    m_cost_sum = df[f'{m}_cost'].sum()
    summary.append({
        'Model': m,
        'Mean Score': df[f'{m}_score'].mean(),
        'Total Cost Ratio': m_cost_sum / base_total_cost,
        'Avg Cost / Query': df[f'{m}_cost'].mean()
    })

summary_df = pd.DataFrame(summary)
print("\n=== 모델별 전체 요약 통계 ===")
print(summary_df.to_string(index=False))

ax31-light 총 비용(Base Total Cost): 5,348,918.00

=== 모델별 전체 요약 통계 ===
     Model  Mean Score  Total Cost Ratio  Avg Cost / Query
ax31-light    0.597301          1.000000       3039.157955
      ax31    0.678551          2.004789       6092.869318
axk1-think    0.811648          9.886571      30046.852273


In [3]:
df['gain_ax31'] = df['ax31_score'] - df['ax31-light_score']
df['gain_think'] = df['axk1-think_score'] - df['ax31-light_score']

df['delta_cost_ax31'] = df['ax31_cost'] - df['ax31-light_cost']
df['delta_cost_think'] = df['axk1-think_cost'] - df['ax31-light_cost']

df['efficiency_ax31'] = df['gain_ax31'] / (df['delta_cost_ax31'] + 1e-8)
df['efficiency_think'] = df['gain_think'] / (df['delta_cost_think'] + 1e-8)


df['type'] = 'D (No Gain)'

df.loc[df['ax31-light_score'] >= 0.9, 'type'] = 'A (Easy - Light Enough)'

df.loc[(df['gain_think'] >= 0.3) & (df['type'] == 'D (No Gain)'), 'type'] = 'B (Hard - Need Think)'

df.loc[(df['gain_ax31'] >= 0.2) & (df['type'] == 'D (No Gain)'), 'type'] = 'C (Medium - Need ax31)'


print("=== [2단계 완료] 문항 유형별 비율 분포 (%) ===")
type_counts = df['type'].value_counts(normalize=True) * 100
type_df = pd.DataFrame({'Type': type_counts.index, 'Percentage (%)': type_counts.values.round(2)})
print(type_df.to_string(index=False))

print("\n=== 유형별 평균 점수 및 비용 현황 ===")
summary_by_type = df.groupby('type').agg(
    count=('episode_id', 'count'),
    avg_score_light=('ax31-light_score', 'mean'),
    avg_score_think=('axk1-think_score', 'mean'),
    avg_gain_think=('gain_think', 'mean')
).reset_index()

print(summary_by_type.to_string(index=False))

=== [2단계 완료] 문항 유형별 비율 분포 (%) ===
                   Type  Percentage (%)
A (Easy - Light Enough)           54.20
  B (Hard - Need Think)           31.82
            D (No Gain)           10.57
 C (Medium - Need ax31)            3.41

=== 유형별 평균 점수 및 비용 현황 ===
                   type  count  avg_score_light  avg_score_think  avg_gain_think
A (Easy - Light Enough)    954         1.000000         0.940514       -0.059486
  B (Hard - Need Think)    560         0.123214         0.925893        0.802679
 C (Medium - Need ax31)     60         0.237500         0.108333       -0.129167
            D (No Gain)    186         0.075269         0.033602       -0.041667


In [8]:
score_stats = []
for m in MODELS:
    scores = df[f'{m}_score']
    score_stats.append({
        'Model': m,
        'Mean Score': scores.mean(),
        'Std Dev': scores.std(),
        'Median (Q2)': scores.median(),
        '25% (Q1)': scores.quantile(0.25),
        '75% (Q3)': scores.quantile(0.75),
        'Perfect (1.0) Ratio (%)': (scores == 1.0).mean() * 100,
        'Zero (0.0) Ratio (%)': (scores == 0.0).mean() * 100
    })

score_summary_df = pd.DataFrame(score_stats)
print("=== 1. 모델별 품질 점수 상세 분포 ===")
print(score_summary_df.to_string(index=False))


cost_stats = []
for m in MODELS:
    p_tokens = df[f'{m}_input_tokens']
    c_tokens = df[f'{m}_output_tokens']
    tot_costs = df[f'{m}_cost']
    
    cost_stats.append({
        'Model': m,
        'Avg Input Tokens': p_tokens.mean(),
        'Avg Output Tokens': c_tokens.mean(),
        'Avg Total Cost / Query': tot_costs.mean(),
        'Cost Std Dev': tot_costs.std(),
        'Max Cost / Query': tot_costs.max(),
        'Cost / 1.0 Score Pt': tot_costs.sum() / (df[f'{m}_score'].sum() + 1e-8)
    })

cost_summary_df = pd.DataFrame(cost_stats)
print("\n=== 2. 모델별 토큰 소비 및 비용 상세 분포 ===")
print(cost_summary_df.to_string(index=False))


print("\n=== 3. ax31-light 대비 모델별 성능/비용 증가율 ===")
base_score = df['ax31-light_score'].mean()
base_cost = df['ax31-light_cost'].mean()

for m in ['ax31', 'axk1-think']:
    m_score = df[f'{m}_score'].mean()
    m_cost = df[f'{m}_cost'].mean()
    
    score_inc = ((m_score - base_score) / (base_score + 1e-8)) * 100
    cost_inc = ((m_cost - base_cost) / (base_cost + 1e-8)) * 100
    
    print(f"[{m}]")
    print(f"  - 평균 점수 상승: +{m_score - base_score:.4f}점 ({score_inc:+.2f}%)")
    print(f"  - 평균 비용 증가: +{m_cost - base_cost:.4f} ({cost_inc:+.2f}%)")

=== 1. 모델별 품질 점수 상세 분포 ===
     Model  Mean Score  Std Dev  Median (Q2)  25% (Q1)  75% (Q3)  Perfect (1.0) Ratio (%)  Zero (0.0) Ratio (%)
ax31-light    0.597301 0.462124          1.0       0.0       1.0                54.204545             34.715909
      ax31    0.678551 0.442920          1.0       0.0       1.0                63.409091             27.613636
axk1-think    0.811648 0.364488          1.0       1.0       1.0                77.045455             14.772727

=== 2. 모델별 토큰 소비 및 비용 상세 분포 ===
     Model  Avg Input Tokens  Avg Output Tokens  Avg Total Cost / Query  Cost Std Dev  Max Cost / Query  Cost / 1.0 Score Pt
ax31-light       2422.705682         616.452273             3039.157955   7556.854808           37363.0          5088.150297
      ax31       2411.027273         635.407386             6092.869318  14972.884583           75040.0          8979.233829
axk1-think       2266.348864        3743.021591            30046.852273  56200.416543          655360.0         37019

In [12]:
import json

with open("data/train/outcomes.json", "r", encoding="utf-8") as f:
    outcomes_sample = json.load(f)

print("1. outcomes 타입:", type(outcomes_sample))

if isinstance(outcomes_sample, list):
    print("2. outcomes 길이:", len(outcomes_sample))
    print("3. 첫 번째 요소 샘플:", outcomes_sample[0])
elif isinstance(outcomes_sample, dict):
    print("2. outcomes keys 샘플:", list(outcomes_sample.keys())[:3])
    first_key = list(outcomes_sample.keys())[0]
    print(
        f"3. 첫 번째 요소({first_key}) 내용 샘플:", outcomes_sample[first_key]
    )

1. outcomes 타입: <class 'dict'>
2. outcomes keys 샘플: ['challenge_id', 'episodes', 'schema_version']
3. 첫 번째 요소(challenge_id) 내용 샘플: ossp-2026-llm-router-challenge


In [15]:
def safe_float(val, default=0.0):
    if val is None:
        return default
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


def safe_int(val, default=0):
    if val is None:
        return default
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default

with open("data/train/inputs-base.json", "r", encoding="utf-8") as f:
    inputs_json = json.load(f)
    inputs_data = (
        inputs_json.get("episodes", inputs_json)
        if isinstance(inputs_json, dict)
        else inputs_json
    )

with open("data/train/outcomes.json", "r", encoding="utf-8") as f:
    outcomes_json = json.load(f)
    outcomes_list = outcomes_json.get("episodes", [])

outcomes_dict = {}
for item in outcomes_list:
    if isinstance(item, dict) and "episode_id" in item:
        outcomes_dict[item["episode_id"]] = item

rows = []

for ep in inputs_data:
    if isinstance(ep, str):
        ep = json.loads(ep)

    ep_id = ep.get("episode_id") if isinstance(ep, dict) else None
    if not ep_id:
        continue

    if "prompt" in ep and ep["prompt"]:
        prompt_text = ep["prompt"]
    else:
        prompt_text = " ".join(
            [m.get("content", "") for m in ep.get("messages", [])]
        )

    ep_outcomes = outcomes_dict.get(ep_id, {})
    if not ep_outcomes:
        continue

    models_info = ep_outcomes.get("models", ep_outcomes)

    light = models_info.get("ax31-light", {})
    std = models_info.get("ax31", {})
    think = models_info.get("axk1-think", {})

    s_light = safe_float(light.get("score"))
    s_std = safe_float(std.get("score"))
    s_think = safe_float(think.get("score"))

    c_light = safe_int(light.get("input_tokens")) + safe_int(
        light.get("output_tokens")
    )
    c_std = safe_int(std.get("input_tokens")) + safe_int(
        std.get("output_tokens")
    )
    c_think = safe_int(think.get("input_tokens")) + safe_int(
        think.get("output_tokens")
    )

    delta_score_think_vs_std = s_think - s_std
    delta_cost_think_vs_std = c_think - c_std

    efficiency_think = delta_score_think_vs_std / (
        delta_cost_think_vs_std + 1e-5
    )

    rows.append(
        {
            "episode_id": ep_id,
            "prompt_len": len(prompt_text),
            "has_code": int(
                "```" in prompt_text
                or "def " in prompt_text
                or "class " in prompt_text
            ),
            "has_math": int(
                "\\" in prompt_text or "$" in prompt_text or "sum" in prompt_text
            ),
            "s_light": s_light,
            "s_std": s_std,
            "s_think": s_think,
            "c_light": c_light,
            "c_std": c_std,
            "c_think": c_think,
            "think_output_tokens": safe_int(think.get("output_tokens")),
            "delta_score": delta_score_think_vs_std,
            "delta_cost": delta_cost_think_vs_std,
            "efficiency": efficiency_think,
        }
    )

df = pd.DataFrame(rows)

if df.empty:
    print("❌ 데이터프레임이 비어 있습니다.")
else:
    type_a = df[df["delta_score"] >= 0.20].sort_values(
        by="efficiency", ascending=False
    )
    type_b = df[(df["delta_score"] < 0.05) & (df["s_std"] >= 0.70)]
    type_c = df[df["s_think"] <= 0.30]

    print(f"=== 분석 결과 요약 (전체 {len(df)}문항) ===")
    print(
        f"1. Type A (Think 필수 고가성비 문항): {len(type_a)}개 ({len(type_a)/len(df)*100:.1f}%)"
    )
    print(
        f"2. Type B (ax31/Light로 충분한 문항): {len(type_b)}개 ({len(type_b)/len(df)*100:.1f}%)"
    )
    print(
        f"3. Type C (Think 써도 점수 낮은 난제): {len(type_c)}개 ({len(type_c)/len(df)*100:.1f}%)"
    )

    if not type_a.empty:
        print("\n--- Type A 문항의 주요 특성 평균 ---")
        print(
            type_a[
                [
                    "prompt_len",
                    "has_code",
                    "has_math",
                    "think_output_tokens",
                    "delta_score",
                    "efficiency",
                ]
            ].mean()
        )

=== 분석 결과 요약 (전체 1736문항) ===
1. Type A (Think 필수 고가성비 문항): 402개 (23.2%)
2. Type B (ax31/Light로 충분한 문항): 1115개 (64.2%)
3. Type C (Think 써도 점수 낮은 난제): 261개 (15.0%)

--- Type A 문항의 주요 특성 평균 ---
prompt_len             5401.378109
has_code                  0.345771
has_math                  0.141791
think_output_tokens    4145.211443
delta_score               0.779229
efficiency                0.000386
dtype: float64
